# Quantum Reasoning Co-Processor — GPU Qubit Scaling Experiment
**Circuit-level test: Does quantum advantage grow with qubit count?**

Author: Saikrishna Garikipati

> **Runtime → Run all** — runs end-to-end unattended. **GPU REQUIRED** (T4/A100).

Tests the QuantumAttentionCircuit (Q-K→V via CNOT) in isolation at 12, 15, 18 qubits.
Compares against classical MLP of matched parameter count on XOR-sign classification.

If advantage grows with qubit count → scaling thesis is alive → proceed to real hardware.

In [ ]:
##############################################################################
# STEP 1: SETUP & GPU VERIFICATION
##############################################################################
import os, sys, time
print('=' * 60)
print('STEP 1: SETUP & GPU VERIFICATION')
print('=' * 60)

!pip install -q pennylane pennylane-lightning 'pennylane-lightning[gpu]' autograd scipy 2>&1 | tail -3

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import autograd
import autograd.numpy as anp
import pennylane as qml
import pennylane.numpy as pnp

print(f'PennyLane {qml.__version__}')

# --- HARD GPU CHECK ---
try:
    test_dev = qml.device('lightning.gpu', wires=4)
    print(f'✓ lightning.gpu AVAILABLE')
    del test_dev
except Exception as e:
    print(f'✗ lightning.gpu NOT available: {e}')
    print()
    print('FATAL: This notebook REQUIRES GPU runtime.')
    print('Fix: Runtime → Change runtime type → GPU (T4 or A100)')
    print('Then: Restart runtime and run all again.')
    raise SystemExit('GPU required. Process killed.')

print('\nSetup complete. GPU ready.')

In [ ]:
##############################################################################
# STEP 2: DEFINE QUANTUM ATTENTION CIRCUIT
##############################################################################
print('=' * 60)
print('STEP 2: QUANTUM ATTENTION CIRCUIT')
print('=' * 60)

def build_attention_circuit(n_qubits, n_layers=2):
    """Build Q-K→V quantum attention circuit on GPU.
    Returns SINGLE SCALAR (mean PauliZ on V-register) for autograd compatibility.
    lightning.gpu + adjoint cannot return multiple expvals through autograd.grad().
    """
    assert n_qubits % 3 == 0
    qpr = n_qubits // 3
    q_wires = list(range(0, qpr))
    k_wires = list(range(qpr, 2 * qpr))
    v_wires = list(range(2 * qpr, 3 * qpr))

    dev = qml.device('lightning.gpu', wires=n_qubits)

    # Hamiltonian: average of PauliZ on V-register → single scalar output
    coeffs = [1.0 / qpr] * qpr
    obs = [qml.PauliZ(w) for w in v_wires]
    H = qml.Hamiltonian(coeffs, obs)

    @qml.qnode(dev, interface='autograd', diff_method='adjoint')
    def circuit(q_features, k_features, v_features, params):
        qml.AmplitudeEmbedding(q_features, wires=q_wires, normalize=True)
        qml.AmplitudeEmbedding(k_features, wires=k_wires, normalize=True)
        qml.AmplitudeEmbedding(v_features, wires=v_wires, normalize=True)
        idx = 0
        for layer in range(n_layers):
            for i in range(qpr):
                qml.CNOT(wires=[q_wires[i], k_wires[i]])
            for i in range(qpr):
                qml.CNOT(wires=[k_wires[i], v_wires[i]])
                qml.RY(params[idx], wires=v_wires[i])
                qml.CNOT(wires=[k_wires[i], v_wires[i]])
                idx += 1
            for i in range(qpr):
                qml.RY(params[idx], wires=v_wires[i])
                qml.RZ(params[idx + 1], wires=v_wires[i])
                idx += 2
            for i in range(qpr - 1):
                qml.CNOT(wires=[v_wires[i], v_wires[i + 1]])
            for i in range(qpr):
                qml.CNOT(wires=[q_wires[i], k_wires[i]])
        return qml.expval(H)

    n_params = n_layers * (qpr + qpr * 2)
    params = pnp.array(np.random.randn(n_params) * 0.5, requires_grad=True)
    print(f'  {n_qubits}q circuit: {n_params} params, register_dim={2**qpr}')
    print(f'  Returns: single scalar (mean PauliZ on V-register)')
    return circuit, params, qpr

# Quick verification
test_circuit, test_params, _ = build_attention_circuit(12)
q_test = np.random.randn(16)
out = test_circuit(q_test, q_test, q_test, test_params)
print(f'  12q forward pass OK: output={float(out):.4f} (scalar in [-1,1])')
print('Circuit definition ready.')

In [ ]:
##############################################################################
# STEP 3: DEFINE CLASSICAL BASELINE
##############################################################################
print('=' * 60)
print('STEP 3: CLASSICAL BASELINE (matched params)')
print('=' * 60)

def build_classical_model(input_dim, output_dim, n_quantum_params):
    """Classical MLP matched to quantum param count."""
    hidden = max(output_dim, n_quantum_params // 3)
    W1 = pnp.array(np.random.randn(input_dim * 3, hidden) * 0.1, requires_grad=True)
    b1 = pnp.array(np.zeros(hidden), requires_grad=True)
    W2 = pnp.array(np.random.randn(hidden, output_dim) * 0.1, requires_grad=True)
    b2 = pnp.array(np.zeros(output_dim), requires_grad=True)
    total = W1.size + b1.size + W2.size + b2.size
    print(f'  Classical: {total} params (hidden={hidden})')
    return [W1, b1, W2, b2]

def classical_forward(q, k, v, params):
    W1, b1, W2, b2 = params
    x = anp.concatenate([q, k, v])
    h = anp.maximum(x @ W1 + b1, 0)
    return anp.tanh(W2.T @ h + b2)

print('Classical model defined.')

In [ ]:
##############################################################################
# STEP 4: DATA GENERATION
##############################################################################
print('=' * 60)
print('STEP 4: XOR-SIGN CLASSIFICATION TASK')
print('=' * 60)

def generate_xor_sign_data(dim, n_samples, seed=42):
    """
    Task: Predict XOR of sign-agreements between Q and K on hidden dims.
    This directly exploits CNOT(Q,K) → |K⊕Q⟩ mechanism.
    """
    rng = np.random.RandomState(seed)
    n_hidden = min(dim // 2, 4)
    hidden_dims = sorted(rng.choice(dim, n_hidden, replace=False).tolist())
    Q = rng.randn(n_samples, dim).astype(np.float64)
    K = rng.randn(n_samples, dim).astype(np.float64)
    V = rng.randn(n_samples, dim).astype(np.float64)
    agreements = np.array([(np.sign(Q[:, i]) == np.sign(K[:, i])).astype(int)
                           for i in hidden_dims])
    labels = np.bitwise_xor.reduce(agreements, axis=0)
    return Q, K, V, labels, hidden_dims

# Verify
Q, K, V, y, hd = generate_xor_sign_data(16, 100, seed=42)
print(f'  Example: dim=16, samples=100, hidden={hd}, balance={y.mean():.2f}')
print('Data generation ready.')

In [ ]:
##############################################################################
# STEP 5: TRAINING FUNCTIONS
##############################################################################
print('=' * 60)
print('STEP 5: TRAINING INFRASTRUCTURE')
print('=' * 60)

def train_quantum_circuit(circuit, params, Q, K, V, y, epochs=30, lr=0.01, batch_size=32):
    n = len(Q)
    for epoch in range(epochs):
        idx = np.random.permutation(n)
        epoch_loss = 0.0
        nb = 0
        for start in range(0, min(n, 200) - batch_size + 1, batch_size):
            batch = idx[start:start + batch_size]
            def loss_fn(p):
                total = 0.0
                for i in batch:
                    # circuit returns a single scalar (mean PauliZ on V-register)
                    score = circuit(Q[i], K[i], V[i], p)
                    pred = (score + 1) / 2  # map [-1,1] to [0,1]
                    t = float(y[i])
                    total = total - (t * anp.log(pred + 1e-10) + (1-t) * anp.log(1-pred + 1e-10))
                return total / len(batch)
            grads = autograd.grad(loss_fn)(params)
            epoch_loss += float(loss_fn(params))
            params = pnp.array(params - lr * np.array(grads), requires_grad=True)
            nb += 1
        if (epoch + 1) % 10 == 0 or epoch == 0:
            acc = eval_quantum(circuit, params, Q[:200], K[:200], V[:200], y[:200])
            print(f'    Epoch {epoch+1:2d}: loss={epoch_loss/max(nb,1):.4f}, acc={acc:.1%}')
    return params

def train_classical_model(params, Q, K, V, y, epochs=30, lr=0.01, batch_size=32):
    n = len(Q)
    for epoch in range(epochs):
        idx = np.random.permutation(n)
        epoch_loss = 0.0
        nb = 0
        for start in range(0, min(n, 200) - batch_size + 1, batch_size):
            batch = idx[start:start + batch_size]
            def loss_fn(flat):
                sizes = [p.size for p in params]
                shapes = [p.shape for p in params]
                ps = []; i = 0
                for s, sh in zip(sizes, shapes):
                    ps.append(flat[i:i+s].reshape(sh)); i += s
                total = 0.0
                for i in batch:
                    out = classical_forward(Q[i], K[i], V[i], ps)
                    score = anp.mean(out)
                    pred = (score + 1) / 2
                    t = float(y[i])
                    total = total - (t * anp.log(pred + 1e-10) + (1-t) * anp.log(1-pred + 1e-10))
                return total / len(batch)
            flat = np.concatenate([np.array(p).flatten() for p in params])
            grads = autograd.grad(loss_fn)(flat)
            epoch_loss += float(loss_fn(flat))
            flat = flat - lr * np.array(grads)
            i = 0
            for j, p in enumerate(params):
                s = p.size
                params[j] = pnp.array(flat[i:i+s].reshape(p.shape), requires_grad=True)
                i += s
            nb += 1
        if (epoch + 1) % 10 == 0 or epoch == 0:
            acc = eval_classical(params, Q[:200], K[:200], V[:200], y[:200])
            print(f'    Epoch {epoch+1:2d}: loss={epoch_loss/max(nb,1):.4f}, acc={acc:.1%}')
    return params

def eval_quantum(circuit, params, Q, K, V, y):
    # circuit returns scalar — positive = class 1, negative = class 0
    correct = sum(1 for i in range(len(Q))
                  if (1 if float(circuit(Q[i], K[i], V[i], params)) > 0 else 0) == int(y[i]))
    return correct / len(Q)

def eval_classical(params, Q, K, V, y):
    correct = sum(1 for i in range(len(Q))
                  if (1 if float(np.mean(classical_forward(Q[i], K[i], V[i], params))) > 0 else 0) == int(y[i]))
    return correct / len(Q)

print('Training functions ready.')

In [ ]:
##############################################################################
# STEP 6: 12-QUBIT EXPERIMENT
##############################################################################
print('=' * 60)
print('STEP 6: 12-QUBIT EXPERIMENT (register_dim=16)')
print('=' * 60)

RESULTS = {}

np.random.seed(42)
dim12 = 2 ** (12 // 3)  # 16
Q12, K12, V12, y12, hd12 = generate_xor_sign_data(dim12, 500, seed=42)
print(f'Data: dim={dim12}, n=500, hidden={hd12}, balance={y12.mean():.2f}')

print('\n  Quantum (12q):')
t0 = time.perf_counter()
circ12, p12, _ = build_attention_circuit(12, n_layers=2)
p12 = train_quantum_circuit(circ12, p12, Q12[:400], K12[:400], V12[:400], y12[:400], epochs=30)
q12_acc = eval_quantum(circ12, p12, Q12[400:], K12[400:], V12[400:], y12[400:])
q12_time = time.perf_counter() - t0
print(f'  Test acc: {q12_acc:.1%} ({q12_time:.1f}s)')

print('\n  Classical (matched):')
t0 = time.perf_counter()
cp12 = build_classical_model(dim12, 12//3, p12.size)
cp12 = train_classical_model(cp12, Q12[:400], K12[:400], V12[:400], y12[:400], epochs=30)
c12_acc = eval_classical(cp12, Q12[400:], K12[400:], V12[400:], y12[400:])
c12_time = time.perf_counter() - t0
print(f'  Test acc: {c12_acc:.1%} ({c12_time:.1f}s)')

RESULTS[12] = {'quantum': q12_acc, 'classical': c12_acc, 'advantage': q12_acc - c12_acc}
print(f'\n  12q ADVANTAGE: {(q12_acc - c12_acc)*100:+.1f}%')

In [ ]:
##############################################################################
# STEP 7: 15-QUBIT EXPERIMENT
##############################################################################
print('=' * 60)
print('STEP 7: 15-QUBIT EXPERIMENT (register_dim=32)')
print('=' * 60)

np.random.seed(42)
dim15 = 2 ** (15 // 3)  # 32
Q15, K15, V15, y15, hd15 = generate_xor_sign_data(dim15, 500, seed=42)
print(f'Data: dim={dim15}, n=500, hidden={hd15}, balance={y15.mean():.2f}')

print('\n  Quantum (15q):')
t0 = time.perf_counter()
circ15, p15, _ = build_attention_circuit(15, n_layers=2)
p15 = train_quantum_circuit(circ15, p15, Q15[:400], K15[:400], V15[:400], y15[:400], epochs=30)
q15_acc = eval_quantum(circ15, p15, Q15[400:], K15[400:], V15[400:], y15[400:])
q15_time = time.perf_counter() - t0
print(f'  Test acc: {q15_acc:.1%} ({q15_time:.1f}s)')

print('\n  Classical (matched):')
t0 = time.perf_counter()
cp15 = build_classical_model(dim15, 15//3, p15.size)
cp15 = train_classical_model(cp15, Q15[:400], K15[:400], V15[:400], y15[:400], epochs=30)
c15_acc = eval_classical(cp15, Q15[400:], K15[400:], V15[400:], y15[400:])
c15_time = time.perf_counter() - t0
print(f'  Test acc: {c15_acc:.1%} ({c15_time:.1f}s)')

RESULTS[15] = {'quantum': q15_acc, 'classical': c15_acc, 'advantage': q15_acc - c15_acc}
print(f'\n  15q ADVANTAGE: {(q15_acc - c15_acc)*100:+.1f}%')

In [ ]:
##############################################################################
# STEP 8: 18-QUBIT EXPERIMENT
##############################################################################
print('=' * 60)
print('STEP 8: 18-QUBIT EXPERIMENT (register_dim=64)')
print('=' * 60)

np.random.seed(42)
dim18 = 2 ** (18 // 3)  # 64
Q18, K18, V18, y18, hd18 = generate_xor_sign_data(dim18, 500, seed=42)
print(f'Data: dim={dim18}, n=500, hidden={hd18}, balance={y18.mean():.2f}')

print('\n  Quantum (18q):')
t0 = time.perf_counter()
circ18, p18, _ = build_attention_circuit(18, n_layers=2)
p18 = train_quantum_circuit(circ18, p18, Q18[:400], K18[:400], V18[:400], y18[:400], epochs=30)
q18_acc = eval_quantum(circ18, p18, Q18[400:], K18[400:], V18[400:], y18[400:])
q18_time = time.perf_counter() - t0
print(f'  Test acc: {q18_acc:.1%} ({q18_time:.1f}s)')

print('\n  Classical (matched):')
t0 = time.perf_counter()
cp18 = build_classical_model(dim18, 18//3, p18.size)
cp18 = train_classical_model(cp18, Q18[:400], K18[:400], V18[:400], y18[:400], epochs=30)
c18_acc = eval_classical(cp18, Q18[400:], K18[400:], V18[400:], y18[400:])
c18_time = time.perf_counter() - t0
print(f'  Test acc: {c18_acc:.1%} ({c18_time:.1f}s)')

RESULTS[18] = {'quantum': q18_acc, 'classical': c18_acc, 'advantage': q18_acc - c18_acc}
print(f'\n  18q ADVANTAGE: {(q18_acc - c18_acc)*100:+.1f}%')

In [ ]:
##############################################################################
# STEP 9: FINAL SUMMARY & SCALING VERDICT
##############################################################################
print('=' * 60)
print('FINAL SUMMARY: QUBIT SCALING RESULTS')
print('=' * 60)
print()
print(f'{"Qubits":<8} {"Reg Dim":<10} {"Quantum":<12} {"Classical":<12} {"Advantage":<12}')
print('-' * 54)
for nq in sorted(RESULTS.keys()):
    r = RESULTS[nq]
    dim = 2 ** (nq // 3)
    print(f'{nq:<8} {dim:<10} {r["quantum"]*100:.1f}%       {r["classical"]*100:.1f}%       {r["advantage"]*100:+.1f}%')

print()
print('=' * 60)
print('SCALING VERDICT')
print('=' * 60)

advantages = [RESULTS[nq]['advantage'] for nq in sorted(RESULTS.keys())]
if len(advantages) >= 2:
    if advantages[-1] > advantages[0] + 0.03:
        print('✓ ADVANTAGE GROWS WITH QUBITS — Scaling thesis VALIDATED')
        print('  → Proceed to Month 2: real quantum hardware')
    elif advantages[-1] > 0.05:
        print('~ ADVANTAGE EXISTS at high qubits but not clearly growing')
        print('  → Worth investigating further with more epochs/seeds')
    elif all(a < 0.03 for a in advantages):
        print('✗ NO ADVANTAGE at any qubit scale')
        print('  → Circuit design may need fundamental rethinking')
    else:
        print('? MIXED RESULTS — no clear trend')
        print('  → Need more seeds and possibly different tasks')

print()
print('Copyright (c) 2026 Saikrishna Garikipati. All Rights Reserved.')